# Practical Pulsar Timing 2: Noise, Whitening, and Gaussianity

Notebook 1 ended with a clean fit: white residuals at the level of the TOA
uncertainties. Real millisecond pulsars are rarely that polite. This notebook
is about what to do when the leftover structure in your residuals is
**stochastic** — noise — rather than a missing deterministic parameter.

### What you will do

1. Simulate a pulsar with **red noise**, watch a naive white-noise fit fail,
   and quantify the failure with an **Anderson–Darling test**.
2. Fit with a proper noise model (generalised least squares), extract the
   **noise realisation**, and produce **whitened residuals** that pass the
   Gaussianity test.
3. See how ignoring noise makes parameter uncertainties **overconfident**.
4. Add a second, *chromatic* process — **DM noise** — and use dual-band data
   to separate it from achromatic red noise.

All data are simulated on the fly with PINT and fitted with JUG; the notebook
is fully self-contained (it does not require notebook 1 to have been run).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from io import StringIO

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

import pint.logging
pint.logging.setup(level="WARNING")

from pint.models import get_model
from pint.simulation import make_fake_toas_uniform

# Configure JAX (CPU, float64) BEFORE it is first imported — see notebook 1.
import os, sys
if "jax" in sys.modules:
    raise RuntimeError("JAX already imported — restart the kernel and rerun from the top")
os.environ["JAX_PLATFORMS"] = "cpu"

from jug.utils.jax_setup import ensure_jax_x64
ensure_jax_x64()
from jug.engine.session import TimingSession

import pathlib
datadir = pathlib.Path("fake_data"); datadir.mkdir(exist_ok=True)

## 1. Red noise: realise it, subtract it, test it

Real millisecond pulsars are not white. Their residuals contain:

- **White noise** beyond the radiometer errors — parameterised per backend as
  `EFAC` (error scaling), `EQUAD` (added in quadrature), `ECORR`
  (epoch-correlated).
- **Red noise** — a stochastic, steep-spectrum process (spin noise, unmodelled
  DM variations, ...) usually modelled as a Fourier basis with a power-law
  spectrum
  $$ P(f) \propto A^2 \left(\frac{f}{f_{\rm yr}}\right)^{-\gamma}. $$

This matters enormously for gravitational-wave searches: the GW background is
*itself* a red process, so a wrong noise model is the difference between a
detection and an artefact.

When we fit with a noise model, the fitter solves a **generalised
least-squares** problem: the Fourier coefficients of the red process are
estimated *jointly* with the timing parameters. That means we can:

1. extract the **maximum-a-posteriori realisation** of the noise — the model's
   best reconstruction of the red wiggle in our data;
2. subtract it to get **whitened residuals**;
3. **test** whether what remains is actually Gaussian with the claimed
   uncertainties — here, with an **Anderson–Darling test**.

### 1a. Simulate a pulsar with red noise

We inject a power-law red process ($\log_{10} A = -13$, $\gamma = 3.5$,
30 frequencies) on top of 1 $\mu$s white noise, using PINT's simulator, over
5 years.


In [ ]:
RN_PAR = """
PSR              J0437-SIM
RAJ              04:37:15.9
DECJ             -47:15:09.1
POSEPOCH         58900
F0               173.687946    1
F1               -1.728e-15    1
PEPOCH           58900
DM               2.64
DMEPOCH          58900
EPHEM            DE440
CLOCK            TT(BIPM2019)
UNITS            TDB
TZRMJD           58900
TZRSITE          meerkat
TZRFRQ           1284
TNRedAmp         -13.0
TNRedGam         3.5
TNRedC           30
"""

np.random.seed(1)
m_rn = get_model(StringIO(RN_PAR))
toas_rn = make_fake_toas_uniform(
    startMJD=58500, endMJD=59800, ntoas=300,
    model=m_rn, obs="meerkat", freq=1284 * u.MHz,
    error=1.0 * u.us, add_noise=True, add_correlated_noise=True,
)
print(f"Simulated {len(toas_rn)} TOAs with injected red noise")

toas_rn.write_TOA_file(datadir / "J0437-SIM.tim", format="tempo2")
(datadir / "J0437-SIM.par").write_text(RN_PAR)

# A par file for the "naive" timer who does not believe in red noise
NAIVE_PAR = "\n".join(l for l in RN_PAR.splitlines() if not l.startswith("TNRed"))
(datadir / "J0437-SIM_naive.par").write_text(NAIVE_PAR)

### 1b. The naive fit: pretend the noise is white

First, fit spin-down only, ignoring the red process entirely — an ordinary
weighted least-squares fit.


In [ ]:
sess_naive = TimingSession(datadir / "J0437-SIM_naive.par",
                           datadir / "J0437-SIM.tim", verbose=False)
fit_naive = sess_naive.fit_parameters(max_iter=10, verbose=False)

res_naive = fit_naive["residuals_us"]
mjd_rn = np.array([t.mjd_int + t.mjd_frac for t in sess_naive.toas_data])
err_rn = np.array([t.error_us for t in sess_naive.toas_data])

plt.figure(figsize=(10, 3.8))
plt.errorbar(mjd_rn, res_naive, yerr=err_rn, fmt=".", ms=3, alpha=0.6)
plt.axhline(0, color="grey", ls="--", lw=0.5)
plt.xlabel("MJD"); plt.ylabel("Residual ($\\mu$s)")
plt.title(f"Naive white-noise fit   (wRMS = "
          f"{sess_naive.weighted_rms(res_naive):.3f} $\\mu$s, errors are 1 $\\mu$s)")
plt.grid(alpha=0.4); plt.show()

The residuals wander coherently on ~year timescales — classic red noise. F0 and
F1 have soaked up the lowest frequencies (a quadratic), but everything else
remains. Let's quantify "these residuals are not what my error bars claim" with
an **Anderson–Darling (A–D) test** on the normalised residuals
$z_i = R_i / \sigma_i$.

The A–D statistic measures the distance between the empirical CDF of the data
and a reference distribution, with extra weight in the tails — which is exactly
where timing outliers live, and why it is preferred over, e.g.,
Kolmogorov–Smirnov.

One subtlety worth knowing. `scipy.stats.anderson` *estimates* the mean and
variance from the data before testing, so it only checks the **shape** of the
distribution — and red noise is itself a Gaussian process, so it will happily
pass that test with an inflated variance! Our null hypothesis is stronger:
if the model and the error bars are both right, $z_i$ should be draws from a
**standard normal**, $\mathcal{N}(0,1)$, mean *and* variance included. For that
fully specified null we compute the A–D statistic directly,

$$ A^2 = -n - \frac{1}{n}\sum_{i=1}^{n} (2i-1)\,
   \big[ \ln \Phi(z_{(i)}) + \ln (1 - \Phi(z_{(n+1-i)})) \big], $$

whose 5% critical value is 2.492 (the sample must not be standardised first —
that is the point).


In [ ]:
from scipy.stats import anderson, norm

AD_CRIT5 = 2.492   # 5% critical value for a fully specified distribution

def ad_n01(z):
    '''Anderson-Darling statistic against a fixed N(0,1) - no parameters estimated.'''
    z = np.sort(np.asarray(z))
    n = len(z)
    cdf = np.clip(norm.cdf(z), 1e-15, 1 - 1e-15)
    i = np.arange(1, n + 1)
    return -n - np.mean((2 * i - 1) * (np.log(cdf) + np.log(1 - cdf[::-1])))

def ad_report(residuals_us, errors_us, label):
    z = residuals_us / errors_us
    z = z - z.mean()              # a constant offset is absorbed by the phase anyway
    shape_stat = anderson(z, dist="norm").statistic   # shape-only test
    full_stat = ad_n01(z)                             # N(0,1) test
    verdict = "PASS" if full_stat < AD_CRIT5 else "FAIL"
    print(f"{label:15s} std = {z.std():5.2f} | A-D shape-only = {shape_stat:5.2f} | "
          f"A-D vs N(0,1) = {full_stat:7.2f}  (5% crit {AD_CRIT5})  -> {verdict}")
    return z, full_stat

z_naive, ad_naive = ad_report(res_naive, err_rn, "Naive fit:")

Compare the two statistics. The *shape-only* test is only mildly perturbed —
red noise is itself a Gaussian process, so the residual *distribution* stays
close to Gaussian-shaped even when the timing model is badly wrong. Against the
claimed $\mathcal{N}(0,1)$, however, the test fails by orders of magnitude: the
scatter is nearly twice what the error bars promise. The noise is real; our
model for it is missing.


### 1c. The proper fit: model the red noise

Now fit with the red-noise model included (the `TNRedAmp/TNRedGam/TNRedC`
lines in the par file). JUG detects the noise model automatically, builds the
Fourier basis, and solves the generalised least-squares problem. The fit result
contains the **noise realisations** — per-TOA reconstructions of each noise
process.


In [ ]:
sess_rn = TimingSession(datadir / "J0437-SIM.par",
                        datadir / "J0437-SIM.tim", verbose=False)
fit_rn = sess_rn.fit_parameters(max_iter=10, verbose=False)

res_rn = fit_rn["residuals_us"]
noise = fit_rn["noise_realizations"]
print("Noise realisations available:", [k for k in noise if not k.endswith("_err")])

total_noise = np.zeros_like(res_rn)
for key in noise:
    if not key.endswith("_err"):
        total_noise += noise[key]

whitened = res_rn - total_noise
print(f"\nPost-fit wRMS : {sess_rn.weighted_rms(res_rn):8.3f} us")
print(f"Whitened wRMS : {sess_rn.weighted_rms(whitened):8.3f} us")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)

axes[0].errorbar(mjd_rn, res_rn, yerr=err_rn, fmt=".", ms=3, alpha=0.6,
                 label="post-fit residuals")
axes[0].plot(mjd_rn, noise["RedNoise"], "-", color="crimson", lw=1.5,
             label="MAP red-noise realisation")
axes[0].set_ylabel("Residual ($\\mu$s)")
axes[0].set_title("The fitter reconstructs the red process it fitted through")
axes[0].legend()

axes[1].errorbar(mjd_rn, whitened, yerr=err_rn, fmt=".", ms=3, alpha=0.6,
                 color="seagreen")
axes[1].axhline(0, color="grey", ls="--", lw=0.5)
axes[1].set_xlabel("MJD"); axes[1].set_ylabel("Residual ($\\mu$s)")
axes[1].set_title(f"Whitened residuals   (wRMS = "
                  f"{sess_rn.weighted_rms(whitened):.3f} $\\mu$s)")
for ax in axes: ax.grid(alpha=0.4)
plt.tight_layout(); plt.show()

### 1d. Gaussianity check on the whitened residuals

If our noise model is adequate, the whitened, normalised residuals should be
draws from $\mathcal{N}(0, 1)$.


In [ ]:
z_white, ad_white = ad_report(whitened, err_rn, "Whitened fit:")
_ = ad_report(res_naive, err_rn, "Naive fit:")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
x = np.linspace(-6, 6, 300)
for ax, z, ad_stat, name, c in [
        (axes[0], z_naive, ad_naive, "Naive fit", "indianred"),
        (axes[1], z_white, ad_white, "Whitened", "seagreen")]:
    ax.hist(z, bins=40, density=True, alpha=0.65, color=c)
    ax.plot(x, norm.pdf(x), "k-", lw=1.4, label="$\\mathcal{N}(0,1)$")
    ax.set_title(f"{name}:  A–D vs $\\mathcal{{N}}(0,1)$ = {ad_stat:.2f}  "
                 f"({'FAIL' if ad_stat > AD_CRIT5 else 'PASS'} at 5%)")
    ax.set_xlabel("Normalised residual ($\\sigma$)")
    ax.legend()
axes[0].set_ylabel("Density")
plt.tight_layout(); plt.show()

The naive fit fails the Anderson–Darling test spectacularly; after modelling
and subtracting the red process, the residuals are consistent with unit
Gaussian. (The whitened scatter comes out very slightly *below* 1 — the MAP
realisation inevitably absorbs a little of the white noise too. With a badly
over-flexible noise model this over-whitening becomes severe; watch for
suspiciously small whitened wRMS.) This "fit → realise → whiten → test" loop is
the everyday hygiene of precision timing.

One more lesson hides in the parameter uncertainties:


In [ ]:
for name in ["F0", "F1"]:
    print(f"{name}:")
    print(f"   naive (white) fit : {fit_naive['final_params'][name]:.15e} "
          f"+/- {fit_naive['uncertainties'][name]:.2e}")
    print(f"   red-noise GLS fit : {fit_rn['final_params'][name]:.15e} "
          f"+/- {fit_rn['uncertainties'][name]:.2e}")
truth = {"F0": 173.687946, "F1": -1.728e-15}
print("\nTruth: F0 = 173.687946, F1 = -1.728e-15")

The naive fit reports *smaller* error bars — it is **overconfident**, because
it assumed every deviation was independent white scatter. The GLS uncertainties
are larger and honest: low-frequency noise is partially degenerate with F0/F1,
and the error bars must reflect that. Underestimated timing uncertainties
propagate directly into overstated PTA sensitivities, which is why noise
modelling is not optional.

**Exercises (10 min)**

1. Re-run 1a with a shallower spectrum ($\gamma = 2$) or a stronger amplitude
   ($\log_{10}A = -12.5$). When does the naive fit's F1 land many sigma from
   the truth?
2. Add white-noise mis-modelling: multiply the simulated TOA errors by 2 in the
   tim file (or add an `EFAC` line) and see how the A–D test responds even
   after whitening.
3. JUG can *estimate* noise parameters (rather than fixing them from the par
   file) via MAP/SVI — see `jug.noise.map_estimator.estimate_noise_parameters`.
   Try estimating `TNRedAmp`/`TNRedGam` from the simulated data and compare to
   the injected values.


## 2. Chromatic noise: DM variations in dual-band data

Achromatic red noise delays every observing frequency equally. But the
interstellar medium is not static: the **dispersion measure** varies with
time as the line of sight drifts through the turbulent ionised medium, and
the resulting delay is strongly **chromatic**:

$$ \Delta t_{\rm DM}(t, \nu) = \frac{K \, \mathrm{DM}(t)}{\nu^2}, $$

so a DM fluctuation delays an 816 MHz TOA
$(1284/816)^2 \approx 2.5\times$ more than a 1284 MHz TOA. **This is why we
observe at more than one frequency**: with a single band, DM noise and spin
noise are hopelessly degenerate; with two bands, the $\nu^{-2}$ scaling lets
the fitter separate them.

We simulate a MeerKAT-style campaign observing every epoch simultaneously at
UHF (816 MHz) and L-band (1284 MHz), with *both* an achromatic red process
and a DM-noise process injected (`TNDM*` parameters, the DM analogue of
`TNRed*`).


In [ ]:
DM_PAR = """
PSR              J0437-CHRO
RAJ              04:37:15.9
DECJ             -47:15:09.1
POSEPOCH         58900
F0               173.687946    1
F1               -1.728e-15    1
PEPOCH           58900
DM               2.64          1
DM1              0.0           1
DMEPOCH          58900
EPHEM            DE440
CLOCK            TT(BIPM2019)
UNITS            TDB
TZRMJD           58900
TZRSITE          meerkat
TZRFRQ           1284
TNRedAmp         -13.0
TNRedGam         3.5
TNRedC           30
TNDMAmp          -13.2
TNDMGam          3.0
TNDMC            30
"""

np.random.seed(3)
m_chro = get_model(StringIO(DM_PAR))

# One TOA per band per epoch, same epochs: multi_freqs_in_epoch guarantees the
# two bands share a single realisation of DM(t)
toas_chro = make_fake_toas_uniform(
    startMJD=58500, endMJD=59800, ntoas=300,
    model=m_chro, obs="meerkat", freq=np.array([816, 1284]) * u.MHz,
    multi_freqs_in_epoch=True,
    error=1.0 * u.us, add_noise=True, add_correlated_noise=True,
)
toas_chro.write_TOA_file(datadir / "J0437-CHRO.tim", format="tempo2")
(datadir / "J0437-CHRO.par").write_text(DM_PAR)
print(f"Simulated {len(toas_chro)} TOAs in two bands")

In [ ]:
sess_chro = TimingSession(datadir / "J0437-CHRO.par",
                          datadir / "J0437-CHRO.tim", verbose=False)
fit_chro = sess_chro.fit_parameters(max_iter=10, verbose=False)

res_chro = fit_chro["residuals_us"]
noise_chro = fit_chro["noise_realizations"]
print("Noise realisations:", [k for k in noise_chro if not k.endswith("_err")])

mjd_chro  = np.array([t.mjd_int + t.mjd_frac for t in sess_chro.toas_data])
err_chro  = np.array([t.error_us for t in sess_chro.toas_data])
freq_chro = np.array([t.freq_mhz for t in sess_chro.toas_data])
uhf = freq_chro < 1000

dm_real  = noise_chro["DMNoise"]
red_real = noise_chro["RedNoise"]

# The DM realisation must scale as nu^-2 between the bands:
ratio = np.std(dm_real[uhf]) / np.std(dm_real[~uhf])
print(f"\nDM realisation RMS ratio UHF/L-band: {ratio:.3f} "
      f"(nu^-2 prediction: {(1284/816)**2:.3f})")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)

for sel, name, c in [(uhf, "816 MHz (UHF)", "mediumblue"),
                     (~uhf, "1284 MHz (L-band)", "darkorange")]:
    axes[0].errorbar(mjd_chro[sel], res_chro[sel], yerr=err_chro[sel],
                     fmt=".", ms=3, alpha=0.55, color=c, label=name)
    axes[1].plot(mjd_chro[sel], dm_real[sel], ".", ms=3, alpha=0.7, color=c)
axes[1].plot(mjd_chro, red_real, ".", ms=2, alpha=0.6, color="seagreen",
             label="RedNoise realisation (both bands)")

axes[0].axhline(0, color="grey", ls="--", lw=0.5)
axes[0].set_ylabel("Residual ($\\mu$s)")
axes[0].set_title("Post-fit residuals: the bands separate when DM wanders")
axes[0].legend()
axes[1].axhline(0, color="grey", ls="--", lw=0.5)
axes[1].set_xlabel("MJD"); axes[1].set_ylabel("Realisation ($\\mu$s)")
axes[1].set_title("DM-noise realisation is $\\nu^{-2}$-split; red noise is achromatic")
axes[1].legend()
for ax in axes: ax.grid(alpha=0.4)
plt.tight_layout(); plt.show()

The top panel is the tell-tale sign every pulsar timer learns to spot: when
the two bands **separate** — mirror-imaging each other around zero — the
culprit is chromatic. The bottom panel shows how the GLS fitter dissects it:
the DM realisation carries the $\nu^{-2}$ band split, while the red-noise
realisation is identical in both bands.

The full whitening loop from Section 1 works exactly the same here — subtract
*both* realisations and test:


In [ ]:
whitened_chro = res_chro - dm_real - red_real
z_chro, ad_chro = ad_report(whitened_chro, err_chro, "Whitened (RN+DM):")

**Exercises**

1. Re-simulate with `TNDMAmp -12.8` (stronger DM noise) and single-band
   L-band-only data (drop `multi_freqs_in_epoch` and pass one frequency). Fit
   with both noise processes. How well can the fitter separate red noise from
   DM noise now? Look at the uncertainties on the realisations
   (`noise_chro["DMNoise_err"]`).
2. **Parameter correlations.** The JUG fit result exposes the parameter
   covariance; strong degeneracies (DM vs F0 in single-band data, M2 vs SINI
   in low-inclination binaries) show up as correlation coefficients near ±1.
   Compare the DM uncertainty in the single-band and dual-band fits.
3. The solar wind adds a *third* chromatic process with an annual signature —
   see the [PINT solar wind
   example](https://nanograv-pint.readthedocs.io/en/latest/examples/solar_wind.html).


## Wrap-up

In this notebook you have:

- fitted through **red noise** with a generalised-least-squares noise model,
  extracted the noise **realisation**, produced **whitened residuals**, and
  verified Gaussianity with an **Anderson–Darling test** against a fully
  specified $\mathcal{N}(0,1)$;
- learned why `scipy.stats.anderson` alone is not enough (it only tests the
  *shape* of the distribution — and red noise is Gaussian-shaped);
- seen that ignoring red noise gives you *overconfident* parameter
  uncertainties — the cardinal sin of PTA analysis;
- separated **chromatic (DM) noise** from achromatic red noise using
  dual-band data and the $\nu^{-2}$ scaling.

This "fit → realise → whiten → test" loop is the everyday hygiene of
precision timing, and it scales directly to real PTA data: swap in a real
pulsar's par/tim files and the workflow is identical.

**Further reading and next steps**

- [PINT noise-fitting examples](https://nanograv-pint.readthedocs.io/en/latest/examples/noise-fitting-example.html)
  — estimating EFAC/EQUAD/ECORR from the data rather than fixing them.
- [PINT red/DM/chromatic noise fitting](https://nanograv-pint.readthedocs.io/en/latest/examples/rednoise-fit-example.html).
- [nanograv/pulsar_timing_school](https://github.com/nanograv/pulsar_timing_school)
  — the standard deep-dive into PTA noise formalism and GW analysis, which is
  where the later days of this school pick up.
- JUG's MAP noise estimation
  (`jug.noise.map_estimator.estimate_noise_parameters`) for estimating the
  `TNRed*`/`TNDM*` amplitudes and slopes themselves.
